# ДЗ 3. Случайные величины и распределения

**Курс «Теория вероятностей», ИТМО.** Ноутбук проверяется **автоматически**: после сдачи он запускается целиком («Выполнить все»), и проверяющая программа сверяет ваши переменные и функции с правильными ответами вашего варианта. Максимум — **10 баллов**.

### Как выполнять и сдавать

1. Загрузите ноутбук в Google Colab: [colab.research.google.com](https://colab.research.google.com) → «Файл → Загрузить блокнот» → выберите этот файл. Дальше открывайте его с Google Диска (папка «Colab Notebooks»).
2. В первой ячейке впишите ФИО, группу и **номер варианта** (выдаёт преподаватель, от 1 до 40).
3. Заполните все места с `...` и `# TODO`. **Не переименовывайте** переменные и функции: проверка ищет их по именам.
4. Ответы **вычисляйте кодом**, а не вписывайте числами с калькулятора. Ноутбук будет перезапущен с нуля, поэтому всё нужное должно считаться внутри него.
5. Перед сдачей: «Среда выполнения → Перезапустить сеанс и выполнить все» (Runtime → Restart session and run all). Ноутбук должен отработать без ошибок, а ячейка «Самопроверка» — показать, что все ответы на месте.
6. Скачайте ноутбук: «Файл → Скачать → Скачать .ipynb». Назовите файл `hw03_Фамилия_Имя.ipynb` и отправьте преподавателю.

### Правила

- Используйте только `numpy`, `scipy`, `pandas`, `matplotlib` (всё есть в Colab).
- Ячейка с ошибкой не обнуляет работу: остальные задачи проверяются независимо. Но переменная, которую не удалось вычислить, получает 0 баллов.
- Весь ноутбук должен выполняться быстрее чем за 2 минуты.
- Текстовые ответы («Вывод») автоматически не оцениваются, но преподаватель может спросить вас о них на защите.

In [ ]:
STUDENT = ""     # ← ФИО
GROUP = ""       # ← группа
VARIANT = 0      # ← номер варианта от 1 до 40

### Параметры варианта и данные

Ячейку ниже **не изменяйте**: она задаёт параметры вашего варианта.

In [ ]:
import numpy as np

N_VARIANTS = 40
DIST_NAMES = ["binom", "poisson", "geom", "uniform", "expon", "norm"]


def params(variant):
    """Параметры варианта. Не изменяйте эту функцию."""
    v = int(variant)
    r = np.random.default_rng(500 + v)
    return {
        # задача 1
        "values1": np.array([0, 1, 2, 3, 4]),
        "weights1": r.integers(1, 10, size=5),
        # задача 2: f(x) = c * x**k на [0; b]
        "k2": 1 + v % 3,
        "b2": 1 + (v // 3) % 3,
        # задача 4
        "lam4": 2.5 + 0.25 * (v % 13),
        "scale4": 10 + v,
        # задача 5
        "m5": 2 + v % 3,
        "mean5": 100 + 10 * (v % 7),
        "t5": 200,
    }


def make_data(variant, with_truth=False):
    """Данные варианта для задачи 4. Не изменяйте эту функцию."""
    p = params(variant)
    r = np.random.default_rng(10_000 + int(variant))
    requests_per_minute = r.poisson(p["lam4"], size=600)
    gaps_sec = np.round(r.exponential(p["scale4"], size=800), 2)
    names = [str(x) for x in r.permutation(DIST_NAMES)[:3]]
    mystery = {}
    for label, name in zip("ABC", names):
        if name == "binom":
            s = r.binomial(20, r.uniform(0.2, 0.5), size=2000)
        elif name == "poisson":
            s = r.poisson(r.uniform(3, 8), size=2000)
        elif name == "geom":
            s = r.geometric(r.uniform(0.2, 0.4), size=2000)
        elif name == "uniform":
            a = r.uniform(-5, 5)
            s = r.uniform(a, a + r.uniform(2, 10), size=2000)
        elif name == "expon":
            s = r.exponential(r.uniform(1, 5), size=2000)
        else:
            s = r.normal(r.uniform(10, 50), r.uniform(2, 8), size=2000)
        mystery[label] = s
    if with_truth:
        return dict(zip("ABC", names))
    return requests_per_minute, gaps_sec, mystery


assert 1 <= VARIANT <= N_VARIANTS, "Впишите номер варианта от 1 до 40 в первой ячейке"
P = params(VARIANT)
requests_per_minute, gaps_sec, mystery = make_data(VARIANT)

import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

print(f"Вариант {VARIANT}")
print("Задача 1: значения", P["values1"], " веса", P["weights1"])
print(f"Задача 2: f(x) = c·x^{P['k2']} на [0; {P['b2']}]")
print(f"Задача 4: {len(requests_per_minute)} минут наблюдений, {len(gaps_sec)} интервалов, выборки A, B, C")
print(f"Задача 5: m = {P['m5']} реплик, среднее время ответа {P['mean5']} мс, порог {P['t5']} мс")

---
## Задача 1. Дискретная случайная величина (2 балла)

Случайная величина $X$ принимает значения `P["values1"]` $=0,1,2,3,4$ с вероятностями, **пропорциональными** весам `P["weights1"]`.

**1.1 (0,5 балла).** Найдите ряд распределения: массив `probs1` из 5 вероятностей.

**1.2 (1 балл).** Напишите функцию `cdf_discrete(values, probs, x)`, которая для любого дискретного распределения (значения `values` упорядочены по возрастанию, `probs` — их вероятности) возвращает $F(x)=P(X\le x)$. Аргумент `x` может быть числом или массивом; результат — массив той же формы (для числа — число). Функция будет проверена на других распределениях, не только на вашем.

**1.3 (0,5 балла).** С помощью `cdf_discrete` найдите `p13a` $=P(1<X\le 3)$ и `p13b` $=P(1\le X\le 3)$.

In [ ]:
# 1.1
probs1 = ...   # TODO

# 1.2
def cdf_discrete(values, probs, x):
    # TODO
    ...

# 1.3
p13a = ...     # TODO
p13b = ...     # TODO

print(probs1, p13a, p13b)

In [ ]:
# График ряда и функции распределения (для себя, не оценивается)
xs = np.linspace(-1, 5, 601)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(P["values1"], probs1); axes[0].set_title("Ряд распределения")
axes[1].plot(xs, cdf_discrete(P["values1"], probs1, xs)); axes[1].set_title("F(x) = P(X ≤ x)")
plt.show()

---
## Задача 2. Непрерывная случайная величина (2 балла)

Плотность случайной величины $X$ равна $f(x)=c\,x^{k}$ при $x\in[0;b]$ и $0$ вне отрезка, где $k$ = `P["k2"]`, $b$ = `P["b2"]`.

**2.1 (0,5 балла).** Найдите константу `c2` из условия нормировки.

**2.2 (0,5 балла).** Напишите функцию `F2(x)` — функцию распределения $X$. Она должна правильно работать при любых $x$, в том числе $x<0$ и $x>b$, для числа и для массива.

**2.3 (1 балл).** Найдите `p23` $=P(X>b/2)$ и медиану `median2`.

Формулы выведите на бумаге, а в коде запишите их через `P["k2"]` и `P["b2"]`.

In [ ]:
k, b = P["k2"], P["b2"]

c2 = ...        # TODO

def F2(x):
    # TODO
    ...

p23 = ...       # TODO
median2 = ...   # TODO

print(c2, F2(np.array([-1, b / 2, b, b + 1])), p23, median2)

---
## Задача 3. Генерация случайных величин (2 балла)

Пользоваться готовыми генераторами распределений (`rng.exponential`, `rng.choice`, `stats....rvs` и т. п.) в этой задаче **нельзя**: только равномерные числа `rng.random(...)` и метод обратной функции.

**3.1 (1 балл).** Напишите `sample_task2(n, rng)` — выборку объёма `n` из распределения задачи 2. Здесь `rng` — генератор `np.random.default_rng(...)`, который передаст проверяющая программа; используйте именно его.

**3.2 (1 балл).** Напишите `sample_discrete(values, probs, n, rng)` — выборку объёма `n` из дискретного распределения с рядом `values`, `probs` (массив из элементов `values`).

Проверка: выборки большого объёма сравниваются с теоретическими распределениями (критерий Колмогорова и частоты).

In [ ]:
def sample_task2(n, rng):
    # TODO
    ...

def sample_discrete(values, probs, n, rng):
    # TODO
    ...

test_rng = np.random.default_rng(0)
s2 = sample_task2(50_000, test_rng)
xx = np.linspace(0, P["b2"], 200)
plt.hist(s2, bins=50, density=True, alpha=0.6, label="ваша выборка")
plt.plot(xx, c2 * xx ** P["k2"], "r", label="f(x)"); plt.legend(); plt.show()

sd = sample_discrete(P["values1"], probs1, 50_000, test_rng)
print("частоты:", np.round([np.mean(sd == v) for v in P["values1"]], 3), " ряд:", np.round(probs1, 3))

---
## Задача 4. Распределения в данных (2 балла)

У вас есть данные мониторинга сервиса:

- `requests_per_minute` — число запросов к сервису в каждую из 600 минут;
- `gaps_sec` — 800 интервалов между соседними заказами в интернет-магазине, в секундах.

**4.1 (0,5 балла).** Считая число запросов в минуту распределённым по Пуассону, оцените параметр: `lam_hat` (метод моментов).

**4.2 (0,5 балла).** Сервис выдерживает $C=\lceil\hat\lambda+2\sqrt{\hat\lambda}\,\rceil$ запросов в минуту. По модели $\mathrm{Pois}(\hat\lambda)$ найдите вероятность перегрузки за минуту `p_overload` $=P(N>C)$.

**4.3 (0,5 балла).** Считая интервалы показательными, оцените интенсивность (через среднее) и найдите по модели время `wait_q95`, которое не превышают 95 % интервалов (в секундах).

**4.4 (0,5 балла).** В словаре `mystery` три выборки `"A"`, `"B"`, `"C"`. Каждая взята из одного из распределений: `"binom"`, `"poisson"`, `"geom"`, `"uniform"`, `"expon"`, `"norm"` (все три — разные). Постройте графики, посмотрите на значения, среднее и дисперсию и определите распределения. Ответ — словарь `answer_types`, например `{"A": "norm", "B": "geom", "C": "uniform"}`. Баллы — только за полностью верный ответ.

In [ ]:
lam_hat = ...       # TODO
C = int(np.ceil(lam_hat + 2 * np.sqrt(lam_hat)))
p_overload = ...    # TODO
wait_q95 = ...      # TODO

print(lam_hat, C, p_overload, wait_q95)

In [ ]:
# Исследуйте выборки: гистограммы, describe(), среднее и дисперсию
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (label, s) in zip(axes, mystery.items()):
    ax.hist(s, bins=40); ax.set_title(label)
plt.show()
pd.DataFrame({label: pd.Series(s).describe() for label, s in mystery.items()}).round(3)

In [ ]:
answer_types = {"A": ..., "B": ..., "C": ...}   # TODO

---
## Задача 5. Монте-Карло: запрос к репликам (2 балла)

Запрос одновременно отправляют на $m$ = `P["m5"]` независимых реплик базы данных и ждут ответа **от всех**. Время ответа каждой реплики распределено показательно со средним `P["mean5"]` мс. Время выполнения запроса $T=\max(T_1,\dots,T_m)$.

**5.1 (1 балл).** Выведите формулу и вычислите точно `p5_exact` $=P(T>t)$ для $t$ = `P["t5"]` мс. Подсказка: $\{T\le t\}=\{T_1\le t,\dots,T_m\le t\}$.

**5.2 (1 балл).** Оцените ту же вероятность симуляцией: `p5_mc`, не меньше 100 000 повторений (здесь можно пользоваться `rng.exponential`). Число повторений запишите в `n5`.

In [ ]:
p5_exact = ...   # TODO

rng5 = np.random.default_rng(VARIANT)
n5 = 100_000
p5_mc = ...      # TODO

print(p5_exact, p5_mc)

**Вывод (не оценивается автоматически).** В 3–5 предложениях: какие распределения вы узнали в задаче 4.4 и по каким признакам? Как растёт $P(T>t)$ в задаче 5 при увеличении числа реплик и почему?

*Ваш ответ:* ...

---
## Самопроверка

Ячейка проверяет только, что все ответы **на месте и правильного типа**. Правильность значений она не проверяет.

In [ ]:
def _self_check():
    ok = True
    def need(name, cond, hint):
        nonlocal ok
        try:
            good = bool(cond())
        except Exception as exc:
            good, hint = False, f"{hint} ({type(exc).__name__}: {exc})"
        print(("✅ " if good else "❌ ") + name + ("" if good else " — " + hint))
        ok &= good
    num = lambda v: isinstance(v, (int, float, np.integer, np.floating)) and np.isfinite(v)
    need("STUDENT, VARIANT", lambda: STUDENT.strip() and 1 <= VARIANT <= 40, "заполните первую ячейку")
    need("probs1", lambda: np.asarray(probs1, float).shape == (5,), "массив из 5 чисел")
    need("cdf_discrete", lambda: np.shape(cdf_discrete([0, 1], [0.5, 0.5], np.array([0.0, 2.0]))) == (2,), "возвращает массив для массива x")
    need("p13a, p13b", lambda: num(float(p13a)) and num(float(p13b)), "числа")
    need("c2", lambda: num(float(c2)), "число")
    need("F2", lambda: np.shape(F2(np.array([0.0, 1.0]))) == (2,), "возвращает массив для массива x")
    need("p23, median2", lambda: num(float(p23)) and num(float(median2)), "числа")
    need("sample_task2", lambda: len(sample_task2(10, np.random.default_rng(1))) == 10, "возвращает n значений")
    need("sample_discrete", lambda: len(sample_discrete([1, 2], [0.5, 0.5], 10, np.random.default_rng(1))) == 10, "возвращает n значений")
    need("lam_hat, p_overload, wait_q95", lambda: all(num(float(v)) for v in [lam_hat, p_overload, wait_q95]), "числа")
    need("answer_types", lambda: set(answer_types) == {"A", "B", "C"} and set(answer_types.values()) <= set(DIST_NAMES), "словарь с ключами A, B, C и названиями из списка")
    need("p5_exact, p5_mc, n5", lambda: num(float(p5_exact)) and num(float(p5_mc)) and n5 >= 100_000, "числа, n5 ≥ 100 000")
    print("\nВсё на месте, можно сдавать." if ok else "\nИсправьте отмеченные пункты.")

_self_check()